# Заполнение пропусков: сравнение стратегий

Сравниваем четыре варианта: без импутации (для CatBoost), медиана/мода, KNN, MICE. Логика в `src/imputation.py`. Анализ природы пропусков (MCAR/MAR/MNAR) - в `src/missingness.py`.

Пропуски структурированы по панелям обследования (MAR), поэтому ожидаем, что методы с учетом связей (KNN, MICE) сохранят распределения лучше простой медианы. Целевую переменную не импутируем. Здесь импутация на всей выборке только для оценки сдвига; в обучении импутер обучается на train внутри фолдов.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

from IPython.display import Image, display
from src import imputation, config

variants = imputation.build_all_variants()
print('Сохранены наборы:', list(variants.keys()))

## Сдвиг распределений до и после импутации

KS_p - критерий Колмогорова-Смирнова между исходными наблюдаемыми значениями и столбцом после импутации. Чем выше KS_p, тем меньше искажение. Низкий KS_p (около 0) означает, что импутация сильно изменила распределение.

In [ ]:
comparison = imputation.compare_distributions(variants)
print('Медианный KS_p по стратегиям (выше - лучше):')
print(comparison.groupby('стратегия')['KS_p'].median().round(3))
comparison

## Распределения по показателям с большой долей пропусков

Черная линия - исходные наблюдаемые значения. Медиана дает искусственный пик, KNN и MICE ближе к исходной форме.

In [ ]:
imputation.plot_watch_distributions(variants)
for col in imputation.WATCH_COLS:
    safe = col.replace(' ', '_').replace('/', '_')[:40]
    f = config.FIGURES_DIR / f'impute_{safe}.png'
    if f.exists():
        display(Image(str(f)))